# 04 — Concordância, calibração e decisão go/no-go

**O que faz:** junta `labels_poc.csv` com `blind_map.csv`; mede concordância
regra×humano no limiar atual (kappa com IC bootstrap); controle negativo humano
(self vs. cross-driver); calibra cada limiar de `THRESHOLD_SPEC` por ROC/Youden contra
o rótulo humano (grade 2D para regras com dois limiares); leave-one-track-out; aplica
o critério go/no-go registrado *antes* da rotulagem; produz a tabela-resumo do artigo.

**Consome:** `%run 00_core.ipynb`, `data/poc_threshold_validation/labels_poc.csv`,
`data/poc_threshold_validation/blind_map.csv`,
`data/poc_threshold_validation/results/rules_in_poc.csv` e `criterion_per_rule.csv`
(de `01_gap_closure.ipynb`).

**Produz:** `results/agreement_current_threshold.csv`, `results/calibration_roc.csv`,
`results/calibration_loto.csv`, `results/go_no_go.md`,
`results/table_validation_summary.csv`, e as figuras correspondentes.

**Ordem de execução:** roda por último (depois de `00`-`03`). **Só produz resultados
com ≥100 rótulos reais** (`labeled_at` preenchido em `labels_poc.csv`) -- caso
contrário, para logo após a junção e avisa. Nesta sessão, `labels_poc.csv` tem 0
rótulos (nenhuma rotulagem humana foi feita), então o notebook **para no guard**, como
pedido -- as células de análise abaixo ficam prontas e corretas, mas não foram
executadas com dados reais.

In [1]:
%run 00_core.ipynb
FACTORS = [0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 2.5, 3.0]

PROJECT_ROOT : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing
DATA_DIR     : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation
Tracks       : ['charlotte_roval_2025', 'summit_point']
Drivers      : {'Rodrigo': 'Driver A', 'Tomaz': 'Driver B', 'Morsinaldo': 'Driver C', 'Thallys': 'Driver D', 'Igor': 'Driver E', 'Hilton': 'Driver F'}
5 comparisons: ['B vs. A', 'C vs. B', 'D vs. B', 'E vs. B', 'F vs. B']
Self-comparison label: 'B vs. B (self)'
Descriptor functions loaded. DESC_SPEC = {'PEDAL_ACTIVE_PCT': 5.0, 'MIN_BRAKE_RUN': 5}
align_lap_by_dist / assert_pedal_scale loaded.
12 rules, 17 tunable thresholds, 2 descriptor parameters.
  MRP_DT               =     -0.05
  BRAKE_EFF_RATIO      =      0.85
  TRAIL_RATIO          =       0.7
  LEGACY_DV            =        -2
  LEGACY_BRAKE_GATE    =         5
  BRAKE_DIST_TOL       =     0.005
  OVERBRAKE_DELTA      =        15
  ENTRY_SLOW_REL       =    -0.015
  ENTRY_BRAKE_GATE     =    

OK (60,130 samples, laps 0–12)
──────────────────────────────────────────────────────────────
  Lap Validity Report  (13 laps total)
──────────────────────────────────────────────────────────────
  ✅ Valid   : 9 laps
  🏆 Fastest : Lap 8  (01:19.767)
  ❌ Invalid : 4 laps
     Lap   0  16:39.700  → LapTime=999.7s | GPS=27.0%<100%
     Lap   1  01:22.067  → IQR_outlier
     Lap   3  01:22.783  → IQR_outlier
     Lap  12  01:22.633  → CompletedPct=0.938<0.995 | GPS=94.0%<100%
──────────────────────────────────────────────────────────────
[SCALE] pedal channels confirmed on a 0-100 scale (brake max = 78.5 %)
pair_cache      : 1 comparison-stints, 18 sector pairs.
pair_cache_self : 1 comparison-stints, 18 sector pairs.
Total activations (cross-driver, current scope): 47
Total activations (self, current scope)        : 31

Cache persisted under C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\cache
stint_slopes / pattern_stats / scenario_summary loade

In [2]:
LABELS_PATH = DATA_DIR / "labels_poc.csv"
BLIND_MAP_PATH = DATA_DIR / "blind_map.csv"

labels_raw = pd.read_csv(LABELS_PATH)
blind_map  = pd.read_csv(BLIND_MAP_PATH)

merged_all = labels_raw.merge(blind_map, on="hash", how="inner", suffixes=("", "_bm"))
labeled_mask = merged_all["labeled_at"].notna() & (merged_all["labeled_at"] != "")
merged = merged_all[labeled_mask].copy()

N_LABELED = len(merged)
MIN_LABELS = 100
print(f"{N_LABELED} labeled rows (of {len(labels_raw)} total hashes in labels_poc.csv).")

if N_LABELED < MIN_LABELS:
    print(f"\n[STOP] {N_LABELED} < {MIN_LABELS} required labels. "
          f"Run 03_labeling_tool.ipynb to collect real human labels, then re-run this "
          f"notebook. No further cells in this notebook should be trusted/executed "
          f"until this guard passes.")
    raise RuntimeError(
        f"Only {N_LABELED} labeled rows (< {MIN_LABELS} required) -- "
        f"analysis stopped per prompt.md guard.")
else:
    print("Guard passed -- proceeding with the analysis.")

0 labeled rows (of 36 total hashes in labels_poc.csv).

[STOP] 0 < 100 required labels. Run 03_labeling_tool.ipynb to collect real human labels, then re-run this notebook. No further cells in this notebook should be trusted/executed until this guard passes.


RuntimeError: Only 0 labeled rows (< 100 required) -- analysis stopped per prompt.md guard.

---
## As células abaixo exigem `N_LABELED >= 100` (guard acima). Prontas e revisadas,
mas **não executadas nesta sessão** (0 rótulos reais disponíveis). Rode este notebook
de novo, do início, depois de coletar rótulos reais em `03_labeling_tool.ipynb`.
---

## 1. Concordância no limiar atual

Por regra (as 10 que entraram na rotulagem -- `RULE_BEHAVIOR[rid] is not None`):
matriz de confusão (coluna de `blind_map` = ativação da regra no limiar baseline;
coluna de comportamento em `labels_poc` = rótulo humano), sensibilidade, especificidade,
PPV, Cohen's kappa com IC bootstrap (1000 reamostragens).

In [ ]:
def cohens_kappa(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    po = np.mean(y_true == y_pred)
    p_t1, p_p1 = np.mean(y_true), np.mean(y_pred)
    pe = p_t1 * p_p1 + (1 - p_t1) * (1 - p_p1)
    return np.nan if np.isclose(1 - pe, 0) else (po - pe) / (1 - pe)


def bootstrap_kappa_ci(y_true, y_pred, n_boot=1000, seed=20250813):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    draws = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        draws[i] = cohens_kappa(y_true[idx], y_pred[idx])
    lo, hi = np.nanpercentile(draws, [2.5, 97.5])
    return float(lo), float(hi)


agreement_rows = []
for rid in LABELING_RULES:
    behavior = RULE_BEHAVIOR[rid]
    sub = merged.dropna(subset=[behavior, rid])
    y_true = sub[behavior].astype(int).to_numpy()
    y_pred = sub[rid].astype(int).to_numpy()
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) else np.nan
    kappa = cohens_kappa(y_true, y_pred) if len(sub) > 0 else np.nan
    seed_rid = 20250813 + (hash(rid) % 10000)
    kappa_lo, kappa_hi = bootstrap_kappa_ci(y_true, y_pred, seed=seed_rid) if len(sub) > 1 else (np.nan, np.nan)
    agreement_rows.append({"Rule_ID": rid, "behavior": behavior, "n": len(sub),
                           "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                           "sensitivity": sens, "specificity": spec, "PPV": ppv,
                           "kappa": kappa, "kappa_ci_lo": kappa_lo, "kappa_ci_hi": kappa_hi})

agreement_df = pd.DataFrame(agreement_rows).sort_values("kappa", ascending=False)
agreement_df.to_csv(RESULTS_DIR / "agreement_current_threshold.csv", index=False)
print(f"Saved -> {RESULTS_DIR / 'agreement_current_threshold.csv'}")
display(agreement_df)

## 2. Controle negativo humano

Taxa de comportamentos marcados (qualquer checkbox = 1) nos painéis "B vs. B (self)"
vs. nos demais, entre os rótulos preenchidos.

In [ ]:
merged["any_behavior"] = merged[[RULE_BEHAVIOR[r] for r in LABELING_RULES]].fillna(0).astype(int).max(axis=1)
merged["is_self"] = merged["Comparison"] == SELF["label"]

ctrl_rows = []
for behavior in [RULE_BEHAVIOR[r] for r in LABELING_RULES] + ["any_behavior"]:
    self_rate  = merged.loc[merged["is_self"], behavior].mean()
    cross_rate = merged.loc[~merged["is_self"], behavior].mean()
    ctrl_rows.append({"behavior": behavior, "self_rate": self_rate, "cross_rate": cross_rate})

df_negctrl = pd.DataFrame(ctrl_rows)
print(f"Self pairs: {int(merged['is_self'].sum())} | Cross-driver pairs: {int((~merged['is_self']).sum())}")
display(df_negctrl)

## 3. Calibração (ROC / Youden)

Para cada regra da rotulagem: 1 limiar -> varredura 1D de `f`; 2 limiares -> grade 2D.
Sensibilidade/especificidade contra o rótulo humano correspondente, avaliadas apenas
sobre os pares rotulados (`pairs_rotulados`). θ* = argmax de Youden.

In [ ]:
labeled_keys = set(zip(merged["Track"], merged["Comparison"], merged["Stint"], merged["Sector"]))

def filter_pairs(pc, keys=labeled_keys):
    out = {}
    for key, rows in pc.items():
        track, label, stint = key
        kept = [(sector, A, B, dts) for sector, A, B, dts in rows
                if (track, label, stint, sector) in keys]
        if kept:
            out[key] = kept
    return out

pairs_rotulados = {**filter_pairs(pair_cache), **filter_pairs(pair_cache_self)}
print(f"pairs_rotulados: {len(pairs_rotulados)} comparison-stints, "
      f"{sum(len(v) for v in pairs_rotulados.values())} labeled sector pairs.")


def youden_for_factors(pairs_subset, merged_subset, rid, behavior, factors):
    df_s = evaluate_rules(pairs_subset, thresholds_at(factors))
    sub = df_s[df_s["Rule_ID"] == rid]
    m = sub.merge(merged_subset[["Track", "Comparison", "Stint", "Sector", behavior]],
                  on=["Track", "Comparison", "Stint", "Sector"], how="inner").dropna(subset=[behavior])
    if m.empty:
        return np.nan, np.nan, np.nan, 0
    y_true = m[behavior].astype(int).to_numpy()
    y_pred = m["Fired"].astype(int).to_numpy()
    tp = int(((y_true == 1) & (y_pred == 1)).sum()); fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum()); fp = int(((y_true == 0) & (y_pred == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    youden = (sens + spec - 1) if pd.notna(sens) and pd.notna(spec) else np.nan
    return sens, spec, youden, len(m)


def calibrate_rule_grid(rid, params, pairs_subset, merged_subset, grid=FACTORS):
    behavior = RULE_BEHAVIOR[rid]
    rows = []
    if len(params) == 1:
        p0 = params[0]
        for f in grid:
            sens, spec, youden, n = youden_for_factors(pairs_subset, merged_subset, rid, behavior, {p0: f})
            rows.append({p0: f, "sens": sens, "spec": spec, "youden": youden, "n": n})
    else:
        p0, p1 = params
        for f0 in grid:
            for f1 in grid:
                sens, spec, youden, n = youden_for_factors(pairs_subset, merged_subset, rid, behavior,
                                                            {p0: f0, p1: f1})
                rows.append({p0: f0, p1: f1, "sens": sens, "spec": spec, "youden": youden, "n": n})
    return pd.DataFrame(rows)


calib_rows = []
calib_grids = {}
for rid in RULE_IDS:
    behavior = RULE_BEHAVIOR[rid]
    params = next(r["params"] for r in DRIVING_RULES if r["id"] == rid)
    if behavior is None:
        for p in params:
            calib_rows.append({"Param": p, "Rule_ID": rid, "behavior": None,
                               "two_param": len(params) == 2, "theta_atual": thresholds_at()[p],
                               "theta_star": np.nan, "f_star": np.nan, "youden_f1": np.nan,
                               "youden_fstar": np.nan, "f_stable_lo": np.nan, "f_stable_hi": np.nan,
                               "n_labeled": 0, "skipped_reason": "no human ground truth (behavior=None)"})
        continue

    grid_df = calibrate_rule_grid(rid, params, pairs_rotulados, merged)
    calib_grids[rid] = grid_df
    if grid_df["youden"].notna().sum() == 0:
        for p in params:
            calib_rows.append({"Param": p, "Rule_ID": rid, "behavior": behavior,
                               "two_param": len(params) == 2, "theta_atual": thresholds_at()[p],
                               "theta_star": np.nan, "f_star": np.nan, "youden_f1": np.nan,
                               "youden_fstar": np.nan, "f_stable_lo": np.nan, "f_stable_hi": np.nan,
                               "n_labeled": 0, "skipped_reason": "no labeled pairs for this rule"})
        continue

    best = grid_df.loc[grid_df["youden"].idxmax()]
    max_youden = best["youden"]
    stable = grid_df[grid_df["youden"] >= 0.95 * max_youden] if max_youden > 0 else grid_df.iloc[0:0]

    for p in params:
        f1_row = grid_df[(grid_df[p] == 1.0)] if len(params) == 1 else grid_df[(grid_df[params[0]] == 1.0) & (grid_df[params[1]] == 1.0)]
        youden_f1 = float(f1_row["youden"].iloc[0]) if not f1_row.empty else np.nan
        calib_rows.append({
            "Param": p, "Rule_ID": rid, "behavior": behavior, "two_param": len(params) == 2,
            "theta_atual": thresholds_at()[p], "theta_star": thresholds_at({p: best[p]})[p],
            "f_star": float(best[p]), "youden_f1": youden_f1, "youden_fstar": float(max_youden),
            "f_stable_lo": float(stable[p].min()) if not stable.empty else np.nan,
            "f_stable_hi": float(stable[p].max()) if not stable.empty else np.nan,
            "n_labeled": int(best["n"]), "skipped_reason": None,
        })

df_calibration = pd.DataFrame(calib_rows)
df_calibration.to_csv(RESULTS_DIR / "calibration_roc.csv", index=False)
print(f"Saved -> {RESULTS_DIR / 'calibration_roc.csv'}")
display(df_calibration)

In [ ]:
single_rules = [rid for rid in LABELING_RULES
                if len(next(r["params"] for r in DRIVING_RULES if r["id"] == rid)) == 1]
n = len(single_rules)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.2), sharey=True)
if n == 1:
    axes = [axes]
for ax, rid in zip(axes, single_rules):
    g = calib_grids.get(rid)
    p = next(r["params"] for r in DRIVING_RULES if r["id"] == rid)[0]
    if g is None or g["youden"].isna().all():
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.plot(g[p], g["youden"], marker="o", markersize=3)
    ax.axvline(1.0, color="grey", ls="--", lw=1)
    ax.set_title(rid, fontsize=8)
    ax.set_xlabel(p, fontsize=7)
axes[0].set_ylabel("Youden J")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_calibration_youden_1d.png", dpi=200, bbox_inches="tight")
plt.show()

two_param_rules = [rid for rid in LABELING_RULES
                   if len(next(r["params"] for r in DRIVING_RULES if r["id"] == rid)) == 2]
if two_param_rules:
    fig, axes = plt.subplots(1, len(two_param_rules), figsize=(4 * len(two_param_rules), 3.6))
    if len(two_param_rules) == 1:
        axes = [axes]
    for ax, rid in zip(axes, two_param_rules):
        g = calib_grids.get(rid)
        p0, p1 = next(r["params"] for r in DRIVING_RULES if r["id"] == rid)
        if g is None or g["youden"].isna().all():
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        else:
            piv = g.pivot(index=p1, columns=p0, values="youden")
            sns.heatmap(piv, cmap="rocket_r", ax=ax, cbar_kws={"label": "Youden J"})
        ax.set_title(rid, fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig_calibration_youden_2d.png", dpi=200, bbox_inches="tight")
    plt.show()
print("Saved -> fig_calibration_youden_1d.png (+ _2d.png if any two-threshold rules)")

## 4. Leave-one-track-out

Calibra θ* usando só os pares rotulados de CLT, testa em SPT (e vice-versa). Reporta a
queda de Youden em relação ao θ* calibrado no próprio track de teste.

In [ ]:
loto_rows = []
for rid in LABELING_RULES:
    behavior = RULE_BEHAVIOR[rid]
    params = next(r["params"] for r in DRIVING_RULES if r["id"] == rid)
    for calib_track, test_track in [("CLT", "SPT"), ("SPT", "CLT")]:
        merged_calib = merged[merged["Track"] == calib_track]
        merged_test  = merged[merged["Track"] == test_track]
        keys_calib = set(zip(merged_calib["Track"], merged_calib["Comparison"], merged_calib["Stint"], merged_calib["Sector"]))
        keys_test  = set(zip(merged_test["Track"],  merged_test["Comparison"],  merged_test["Stint"],  merged_test["Sector"]))
        pairs_calib = {**filter_pairs(pair_cache, keys_calib), **filter_pairs(pair_cache_self, keys_calib)}
        pairs_test  = {**filter_pairs(pair_cache, keys_test),  **filter_pairs(pair_cache_self, keys_test)}

        if merged_calib.empty or merged_test.empty:
            loto_rows.append({"Rule_ID": rid, "calibrated_on": calib_track, "tested_on": test_track,
                              "youden_own": np.nan, "youden_cross": np.nan, "drop": np.nan,
                              "reason": "no labeled pairs on one side"})
            continue

        grid_calib = calibrate_rule_grid(rid, params, pairs_calib, merged_calib)
        grid_own   = calibrate_rule_grid(rid, params, pairs_test, merged_test)
        if grid_calib["youden"].notna().sum() == 0 or grid_own["youden"].notna().sum() == 0:
            loto_rows.append({"Rule_ID": rid, "calibrated_on": calib_track, "tested_on": test_track,
                              "youden_own": np.nan, "youden_cross": np.nan, "drop": np.nan,
                              "reason": "no valid grid points"})
            continue

        best_calib = grid_calib.loc[grid_calib["youden"].idxmax()]
        factors_star = {p: float(best_calib[p]) for p in params}
        _, _, youden_cross, n_cross = youden_for_factors(pairs_test, merged_test, rid, behavior, factors_star)
        youden_own = float(grid_own["youden"].max())
        loto_rows.append({"Rule_ID": rid, "calibrated_on": calib_track, "tested_on": test_track,
                          "youden_own": youden_own, "youden_cross": youden_cross,
                          "drop": youden_own - youden_cross if pd.notna(youden_cross) else np.nan,
                          "reason": None})

df_loto = pd.DataFrame(loto_rows)
df_loto.to_csv(RESULTS_DIR / "calibration_loto.csv", index=False)
print(f"Saved -> {RESULTS_DIR / 'calibration_loto.csv'}")
display(df_loto)

## 5. Go/no-go

Critério fixado antes da rotulagem: **≥8 regras com kappa > 0,4 E θ* dentro de
f ∈ [0,5; 1,5] para a maioria dos limiares avaliáveis** (limiares de regras com
`behavior = None` não têm rótulo humano e ficam fora da contagem).

In [ ]:
n_kappa_pass = int((agreement_df["kappa"] > 0.4).sum())

evaluable = df_calibration[df_calibration["skipped_reason"].isna()].copy()
def in_range(row):
    return 0.5 <= row["f_star"] <= 1.5
evaluable["f_star_ok"] = evaluable.apply(in_range, axis=1)
n_theta_ok = int(evaluable["f_star_ok"].sum())
n_evaluable = len(evaluable)
majority_ok = n_theta_ok > n_evaluable / 2 if n_evaluable > 0 else False

GO = (n_kappa_pass >= 8) and majority_ok

lines = [
    "# Go/No-Go -- POC de validacao de construto e calibracao de limiares", "",
    f"**Resultado: {'GO' if GO else 'NO-GO'}**", "",
    f"- Regras com kappa > 0.4: {n_kappa_pass} / {len(agreement_df)} (criterio: >= 8)",
    f"- Limiares avaliaveis com theta* em f in [0.5, 1.5]: {n_theta_ok} / {n_evaluable} "
    f"(criterio: maioria, i.e. > {n_evaluable/2:.1f})",
    "", "## Suporte -- concordancia por regra", "",
    agreement_df[["Rule_ID", "n", "kappa", "kappa_ci_lo", "kappa_ci_hi"]].to_markdown(index=False),
    "", "## Suporte -- theta* por limiar avaliavel", "",
    evaluable[["Param", "Rule_ID", "f_star", "f_star_ok"]].to_markdown(index=False),
]
go_no_go_text = "\n".join(lines)
(RESULTS_DIR / "go_no_go.md").write_text(go_no_go_text, encoding="utf-8")
print(go_no_go_text)
print(f"\nSaved -> {RESULTS_DIR / 'go_no_go.md'}")

## 6. Tabela para o artigo

Uma linha por regra: tipo de validação recebida (construto / critério / ambas /
nenhuma), kappa, Cliff's delta (de `01_gap_closure.ipynb`), θ_atual, θ*, intervalo
estável.

In [ ]:
criterion_path = RESULTS_DIR / "criterion_per_rule.csv"
df_criterion_loaded = pd.read_csv(criterion_path) if criterion_path.exists() else pd.DataFrame()

summary_rows = []
for rid in RULE_IDS:
    behavior = RULE_BEHAVIOR[rid]
    row = {"Rule_ID": rid, "behavior": behavior}

    if not df_criterion_loaded.empty and rid in df_criterion_loaded["Rule_ID"].values:
        crow = df_criterion_loaded.set_index("Rule_ID").loc[rid]
        delta, pval = crow["cliffs_delta"], crow["p_value"]
        row["cliffs_delta"] = delta
        row["criterion_pass"] = bool(pd.notna(delta) and abs(delta) >= 0.1 and pd.notna(pval) and pval <= 0.05)
    else:
        row["cliffs_delta"] = np.nan
        row["criterion_pass"] = False

    if behavior is not None and rid in agreement_df["Rule_ID"].values:
        arow = agreement_df.set_index("Rule_ID").loc[rid]
        row["kappa"] = arow["kappa"]
        row["construct_pass"] = bool(pd.notna(arow["kappa"]) and arow["kappa"] > 0.4)
    else:
        row["kappa"] = np.nan
        row["construct_pass"] = False

    rule_params = df_calibration[df_calibration["Rule_ID"] == rid]
    if not rule_params.empty:
        row["theta_atual"] = "; ".join(f"{p}={v:.4g}" for p, v in zip(rule_params["Param"], rule_params["theta_atual"]))
        row["theta_star"]  = "; ".join(
            f"{p}={v:.4g}" if pd.notna(v) else f"{p}=NA" for p, v in zip(rule_params["Param"], rule_params["theta_star"]))
        row["stable_interval"] = "; ".join(
            f"{p}=[{lo:.2f},{hi:.2f}]" if pd.notna(lo) and pd.notna(hi) else f"{p}=NA"
            for p, lo, hi in zip(rule_params["Param"], rule_params["f_stable_lo"], rule_params["f_stable_hi"]))
    else:
        row["theta_atual"] = row["theta_star"] = row["stable_interval"] = None

    if row["construct_pass"] and row["criterion_pass"]:
        tipo = "ambas"
    elif row["construct_pass"]:
        tipo = "construto"
    elif row["criterion_pass"]:
        tipo = "critério"
    else:
        tipo = "nenhuma"
    row["validation_type"] = tipo

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)[
    ["Rule_ID", "behavior", "validation_type", "kappa", "cliffs_delta",
     "theta_atual", "theta_star", "stable_interval"]
]
df_summary.to_csv(RESULTS_DIR / "table_validation_summary.csv", index=False)
print(f"Saved -> {RESULTS_DIR / 'table_validation_summary.csv'}")
display(df_summary)